In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

SOURCE_DATA_DIR = Path(os.environ["SOURCE_DATA_DIR"])
OUTPUT_DATA_DIR = Path(os.environ["OUTPUT_DATA_DIR"])

# How many categories to show per subject, busiest first. None = all.
TOP_N = None
# Skip categories with too few attempts to form even one block (empty panels).
MIN_ATTEMPTS = 100

# Per-algorithm styling for the ruptures changepoint overlay: a validated
# categorical trio (worst-pair CVD dE 9.2) PLUS a distinct linestyle each, so
# algorithm identity survives colour-blindness and greyscale printing.
RUPTURES_STYLE = {
    "pelt":   {"color": "#eb6834", "linestyle": (0, (1, 1))},        # orange, dotted
    "binseg": {"color": "#4a3aa7", "linestyle": (0, (5, 2))},        # violet, dashed
    "dynp":   {"color": "#1baf7a", "linestyle": (0, (3, 1, 1, 1))},  # aqua, dash-dot
}

DURATION_COLOR = "#2a78d6"    # blue - the signal that reveals learning
COMPLETION_COLOR = "#898781"  # muted grey - the flat near-ceiling contrast
GRID_COLOR = "#e1e0d9"

In [ ]:
import sys
sys.path.insert(0, str(Path("__file__").resolve().parent.parent))
from analysis.changepoints import load_all_attempts, category_attempt_series, combined_timeline_series

In [ ]:
def block_series(values, block_size):
    """Block-averaged series + its attempt-axis (block centres)."""
    values = np.asarray(values, dtype=float)
    n_blocks = len(values) // block_size
    if n_blocks < 1:
        return np.array([]), np.array([])
    blocks = values[:n_blocks * block_size].reshape(n_blocks, block_size).mean(axis=1)
    axis = np.arange(n_blocks) * block_size + block_size // 2
    return blocks, axis


def overlay_ruptures(ax, block_results):
    """Draw one styled vertical line-set per algorithm from a 'blocks' result."""
    for algo, style in RUPTURES_STYLE.items():
        for cp in block_results.get(algo, []):
            ax.axvline(cp, linewidth=1.2, alpha=0.9, label=f"ruptures {algo}", **style)


def dedup_legend(ax, **kw):
    handles, labels = ax.get_legend_handles_labels()
    seen = {}
    for h, l in zip(handles, labels):
        seen.setdefault(l, h)
    ax.legend(seen.values(), seen.keys(), **kw)

In [ ]:
# ── Combined timeline: all attempts on ONE chronological axis ──────────────
# Each attempt's Duration is z-scored against the subject's own attempts at the
# same scene (removing scene-difficulty), then pooled in time order. This turns
# 24 short choppy per-category series into a single session-wide learning signal
# with meaningful changepoints. Saved as its own figure per subject.
cp_files = sorted((OUTPUT_DATA_DIR / "changepoints_by_category").glob("*_changepoints_by_category.json"))

for cp_file in cp_files:
    result = json.loads(cp_file.read_text())
    combined = result.get("combined_timeline")
    if not combined:
        continue
    subject = result["subject"]
    block_size = result["block_size"]

    records = load_all_attempts(subject, SOURCE_DATA_DIR)
    normalized, _raw = combined_timeline_series(records)
    blocks, axis = block_series(normalized, block_size)

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.axhline(0, color=GRID_COLOR, linewidth=1.0, zorder=0)
    ax.plot(axis, blocks, color=DURATION_COLOR, linewidth=1.6,
            label=f"scene-normalized duration (z), per {block_size} attempts")
    overlay_ruptures(ax, combined["results"]["blocks"])

    ax.set_xlim(0, len(normalized))
    ax.set_xlabel("attempt number (whole session, chronological)")
    ax.set_ylabel("duration vs. scene norm  (z-score)\n← faster    slower →")
    ax.grid(True, color=GRID_COLOR, linewidth=0.6, alpha=0.9)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.set_title(f"Subject {subject} — combined learning timeline "
                 f"({combined['n_attempts']} attempts, all categories pooled & scene-detrended)",
                 fontsize=12, fontweight="bold")
    dedup_legend(ax, loc="upper right", fontsize=8, framealpha=0.9)
    fig.tight_layout()

    out_path = OUTPUT_DATA_DIR / "changepoints_by_category" / f"figure_changepoints_combined_{subject}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved → {out_path}")
    plt.show()

In [ ]:
cp_files = sorted((OUTPUT_DATA_DIR / "changepoints_by_category").glob("*_changepoints_by_category.json"))

for cp_file in cp_files:
    result = json.loads(cp_file.read_text())
    subject = result["subject"]
    block_size = result["block_size"]
    categories = result["categories"]

    # rebuild the per-category Duration / completion series for display
    records = load_all_attempts(subject, SOURCE_DATA_DIR)
    by_cat = category_attempt_series(records, SOURCE_DATA_DIR)

    # rank categories by attempt count, busiest first; drop sparse ones (empty
    # panels) and keep TOP_N if set
    ranked = [c for c in sorted(categories, key=lambda c: categories[c]["n_attempts"], reverse=True)
              if categories[c]["n_attempts"] >= MIN_ATTEMPTS]
    if TOP_N is not None:
        ranked = ranked[:TOP_N]
    if not ranked:
        continue

    fig, axes = plt.subplots(len(ranked), 2, figsize=(14, 2.4 * len(ranked)), squeeze=False)
    fig.suptitle(f"Subject {subject} — changepoints by scene-category",
                 fontsize=14, fontweight="bold", y=1.002)

    for row, cat in enumerate(ranked):
        cat_records = by_cat.get(cat, [])
        durations = [float(r.get("Duration", 0.0)) for r in cat_records]
        completions = [1 if r.get("Outcome") == "completed" else 0 for r in cat_records]

        # left: Duration (the signal expected to reveal learning)
        ax_d = axes[row][0]
        d_blocks, d_axis = block_series(durations, block_size)
        ax_d.plot(d_axis, d_blocks, color=DURATION_COLOR, linewidth=1.4,
                  label=f"mean duration per {block_size} attempts")
        overlay_ruptures(ax_d, categories[cat]["duration"]["blocks"])
        ax_d.set_ylabel(f"{cat}\n({categories[cat]['n_attempts']} att)",
                        fontsize=8.5, fontweight="bold")
        if row == 0:
            ax_d.set_title("time-to-complete (seconds)", fontsize=10)

        # right: completion rate (near ceiling — the contrast)
        ax_c = axes[row][1]
        c_blocks, c_axis = block_series(completions, block_size)
        ax_c.plot(c_axis, c_blocks, color=COMPLETION_COLOR, linewidth=1.4,
                  label=f"completion rate per {block_size} attempts")
        overlay_ruptures(ax_c, categories[cat]["completion"]["blocks"])
        ax_c.set_ylim(-0.05, 1.05)
        if row == 0:
            ax_c.set_title("completion rate", fontsize=10)

        for ax in (ax_d, ax_c):
            ax.grid(True, color=GRID_COLOR, linewidth=0.6, alpha=0.9)
            ax.set_axisbelow(True)
            for spine in ("top", "right"):
                ax.spines[spine].set_visible(False)
            dedup_legend(ax, loc="best", fontsize=6.5, framealpha=0.9)

    for ax in axes[-1]:
        ax.set_xlabel("attempt number (within category)")
    fig.tight_layout()

    out_path = OUTPUT_DATA_DIR / "changepoints_by_category" / f"figure_changepoints_by_category_{subject}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved → {out_path}  ({len(ranked)} categories ≥ {MIN_ATTEMPTS} attempts)")
    plt.show()